In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
x, y = make_classification(n_samples=1000, n_features=8, n_informative=5, n_redundant=2, random_state=42)

col_names = [f"feature_{i}" for i in range(1, 9)]

In [3]:
df = pd.DataFrame(x, columns=col_names)
df["target"] = y

df.head()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,target
0,0.243847,-0.164471,-0.705182,-0.015433,-0.078425,0.730461,0.363777,0.727296,0
1,0.650237,0.274936,-0.776536,-0.959132,-1.123291,-0.572504,-0.918580,0.073886,0
2,-3.390672,-0.590640,-1.343470,6.169133,1.222753,1.439649,-1.715076,3.223089,1
3,0.064793,-0.032960,-0.723131,-1.063065,-0.201072,-0.770711,0.890636,-1.379626,1
4,0.438060,-0.914989,0.953398,-0.033631,-0.048063,-0.860909,0.511169,-0.449294,0


_Create a leakage feature_

In [4]:
df["leakage_feature"] = df["target"] + np.random.normal(0, 0.01, size=len(df))

df.head()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,target,leakage_feature
0,0.243847,-0.164471,-0.705182,-0.015433,-0.078425,0.730461,0.363777,0.727296,0,0.012504
1,0.650237,0.274936,-0.776536,-0.959132,-1.123291,-0.572504,-0.918580,0.073886,0,0.006018
2,-3.390672,-0.590640,-1.343470,6.169133,1.222753,1.439649,-1.715076,3.223089,1,0.990677
3,0.064793,-0.032960,-0.723131,-1.063065,-0.201072,-0.770711,0.890636,-1.379626,1,0.991431
4,0.438060,-0.914989,0.953398,-0.033631,-0.048063,-0.860909,0.511169,-0.449294,0,-0.009541


#### Wrong: using leakage_feature in training

In [5]:
x = df.drop("target", axis=1)
y = df["target"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [6]:
wrong_model = Pipeline([
    ("sclaer", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

wrong_model.fit(x_train, y_train)

y_pred_wrong = wrong_model.predict(x_test)
print(accuracy_score(y_test, y_pred_wrong))
print(classification_report(y_test, y_pred_wrong))

1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       100

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



#### Correct: remove leakage feature

In [7]:
x_correct = df.drop(["target", "leakage_feature"], axis=1)
y = df["target"]

x_train, x_test, y_train, y_test = train_test_split(
    x_correct, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [8]:
correct_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])
correct_model.fit(x_train, y_train)

y_pred_correct = correct_model.predict(x_test)
print(accuracy_score(y_test, y_pred_correct))
print(classification_report(y_test, y_pred_correct))

0.77
              precision    recall  f1-score   support

           0       0.75      0.80      0.78       100
           1       0.79      0.74      0.76       100

    accuracy                           0.77       200
   macro avg       0.77      0.77      0.77       200
weighted avg       0.77      0.77      0.77       200

